In [1]:
import nbimporter



In [2]:
# =============================================================================
#
# Pronostico time trend + dummy seasonal
#
# =============================================================================

#
# Carga de datos
#

import functions

df_orig = functions.load_data()
df_orig.head()



,yt_true
date,
1946-01-01,890
1946-02-01,992
1946-03-01,979
1946-04-01,959
1946-05-01,1110


In [3]:

#
# Componentes de tendencia lineal
#
df_orig = functions.add_linear_trend_component(df_orig)
df_orig




,yt_true,trend
date,,
1946-01-01,890,0
1946-02-01,992,1
1946-03-01,979,2
1946-04-01,959,3
1946-05-01,1110,4
...,...,...
1966-08-01,6905,247
1966-09-01,5504,248
1966-10-01,5457,249


In [4]:

#
# Componentes estacionales
#
df_orig = functions.add_month_component(df_orig)
df_orig.head(15)

,yt_true,trend,month
date,,,
1946-01-01,890,0,1
1946-02-01,992,1,2
1946-03-01,979,2,3
1946-04-01,959,3,4
1946-05-01,1110,4,5
1946-06-01,1546,5,6
1946-07-01,1539,6,7
1946-08-01,3401,7,8
1946-09-01,2092,8,9


In [5]:
#
# Particionamiento de los datos
#
(
    X_complete,
    y_complete,
    X_train,
    y_train,
    X_test,
    y_test,
) = functions.train_test_split(
    df=df_orig,
    x_columns=["trend", "month"],
    y_column="yt_true",
)



In [6]:
#
# Especificación del Modelo
#
from sklearn.compose import ColumnTransformer  # type: ignore
from sklearn.linear_model import LinearRegression  # type: ignore
from sklearn.pipeline import Pipeline  # type: ignore
from sklearn.preprocessing import (  # type: ignore
    MinMaxScaler,
    OneHotEncoder,
    PolynomialFeatures,
)


In [7]:

def create_pipeline(degree):
    pipeline = Pipeline(
        [
            (
                "preprocessor",
                ColumnTransformer(
                    [
                        (
                            "trend",
                            Pipeline(
                                [
                                    ("poly", PolynomialFeatures(degree=degree)),
                                    ("scaler", MinMaxScaler()),
                                ]
                            ),
                            ["trend"],
                        ),
                        ("seasonal", OneHotEncoder(), ["month"]),
                    ]
                ),
            ),
            ("regressor", LinearRegression()),
        ]
    )

    return pipeline
  
  
  

In [8]:
  
#
# Entrenamiento y pronostico
#
for degree in range(1, 4):
    pipeline = create_pipeline(degree=degree)
    pipeline.fit(X_train, y_train)
    df_orig[f"yt_pred_time_trend_{degree}_dummy"] = pipeline.predict(X_complete)

functions.plot_time_series(df_orig)  
  
  
  

<Figure size 1200x400 with 1 Axes>

In [9]:
#
# Almacenamiento de los resultados
#
functions.save_forecasts(df_orig)

In [10]:
#
# Métricas de error
#
metrics = functions.compute_evaluation_metrics(df_orig)
functions.save_metrics(metrics)
metrics
  
  

,Metrics,yt_pred_time_trend_1_dummy,yt_pred_time_trend_2_dummy,yt_pred_time_trend_3_dummy
0,MSE Train,103217.67,78075.48,68044.66
1,MSE Test,738282.04,190079.09,107356.45
2,MAE Train,259.03,213.47,202.35
3,MAE Test,804.13,341.66,262.00
